In [0]:
# Databricks notebook source

# COMMAND ----------
# Formula 1 Racing Data Platform
# Notebook: silver\00_utils
#
# Purpose:
# Utility methods for the silver layer processing.

In [0]:
# COMMAND ----------
# Imports

from pyspark.sql import DataFrame
from pyspark.sql.functions import col, trim
from datetime import datetime
from pyspark.sql.types import StringType

In [0]:
# COMMAND ----------
# Remove duplicates from df

def remove_duplicates(df: DataFrame) -> DataFrame:
    """
    Removes duplicate rows.
    """
    return df.dropDuplicates()

In [0]:
# COMMAND ----------
# Trim strings

def trim_string_columns(df: DataFrame) -> DataFrame:
    """
    Applies trim() to every string column.
    """

    for field in df.schema.fields:

        if isinstance(field.dataType, StringType):

            df = df.withColumn(
                field.name,
                trim(col(field.name))
            )

    return df

In [0]:
# COMMAND ----------
# Validate primary key

def validate_primary_key(
    df: DataFrame,
    pk_column: str
) -> bool:
    """
    Returns True if the column is unique and contains no null values.
    """

    total = df.count()

    distinct = (
        df
        .select(pk_column)
        .distinct()
        .count()
    )

    nulls = (
        df
        .filter(col(pk_column).isNull())
        .count()
    )

    return total == distinct and nulls == 0

In [0]:
# COMMAND ----------
# Validate foreign key

def assert_foreign_key(child_df, parent_df, fk_column):

    invalid_df = (
        child_df.join(
            parent_df.select(fk_column).distinct(),
            fk_column,
            "left_anti"
        )
    )

    invalid_count = invalid_df.count()

    assert invalid_count == 0, (
        f"Foreign key validation failed for '{fk_column}'. "
        f"Found {invalid_count} invalid records."
    )

In [0]:
# COMMAND ----------
# Write delta table

def write_delta(
    df: DataFrame,
    table_name: str
):

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            True
        )
        .saveAsTable(table_name)
    )

    print(f"Table {table_name} created successfully.")

In [0]:
# COMMAND ----------
# Pipeline metrics

def pipeline_metrics(
    table_name,
    rows_before,
    rows_after
):

    return {
        "table_name": table_name,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "execution_timestamp": datetime.now()
    }